# 🚀 Notebook 2: YOLOv11 Fine-Tuning for Cattle Detection
**Cattle Vision Pipeline — Texas A&M University**  
**Author:** Sreenivasulu Reddy Settipalli | sreenivasulureddy@tamu.edu

This notebook covers:
1. Loading the pre-trained YOLOv11 backbone
2. Fine-tuning on our cattle dataset
3. Monitoring training metrics (loss, mAP)
4. Evaluating on the validation set
5. Running inference on sample images

In [ ]:
!pip install ultralytics supervision -q
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import pandas as pd
import numpy as np

# Data config — update path to your dataset
DATA_YAML = 'configs/yolo_cattle.yaml'  # or the Roboflow data.yaml path
print('Imports done ✅')

## 1. Load YOLOv11 and Inspect

In [ ]:
# Load YOLOv11-Large (pretrained on COCO)
model = YOLO('yolo11l.pt')
print(model.info())

## 2. Fine-Tune on Cattle Dataset

In [ ]:
# Fine-tune — takes ~2hrs on T4, ~40min on A100
results = model.train(
    data=DATA_YAML,
    epochs=150,
    imgsz=640,
    batch=16,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    patience=30,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,
    project='runs/train',
    name='cattle_yolo11',
    exist_ok=True,
    verbose=True,
)
print('Training complete ✅')
print(f'Best mAP50: {results.results_dict["metrics/mAP50(B)"]:.4f}')

## 3. Plot Training Curves

In [ ]:
results_csv = Path('runs/train/cattle_yolo11/results.csv')
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics = [
    ('train/box_loss', 'Box Loss', 'Train'),
    ('train/cls_loss', 'Class Loss', 'Train'),
    ('metrics/mAP50(B)', 'mAP@50', 'Val'),
    ('metrics/mAP50-95(B)', 'mAP@50-95', 'Val'),
    ('metrics/precision(B)', 'Precision', 'Val'),
    ('metrics/recall(B)', 'Recall', 'Val'),
]
for ax, (col, title, split) in zip(axes.flat, metrics):
    color = '#E74C3C' if split == 'Train' else '#2ECC71'
    ax.plot(df['epoch'], df[col], color=color, linewidth=2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.3)
plt.suptitle('YOLOv11 Training Curves — Cattle Detection', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Curves saved ✅')

## 4. Evaluate on Validation Set

In [ ]:
best_model = YOLO('runs/train/cattle_yolo11/weights/best.pt')
val_results = best_model.val(data=DATA_YAML, split='val', verbose=True)

print('\n=== Validation Results ===')
print(f'mAP@50:     {val_results.box.map50:.4f}')
print(f'mAP@50-95:  {val_results.box.map:.4f}')
print(f'Precision:  {val_results.box.p.mean():.4f}')
print(f'Recall:     {val_results.box.r.mean():.4f}')

# Per-class results
class_names = ['cattle', 'calf', 'bull']
print('\nPer-class mAP@50:')
for name, ap in zip(class_names, val_results.box.ap50):
    print(f'  {name:10s}: {ap:.4f}')

## 5. Inference on Sample Images

In [ ]:
import cv2
import supervision as sv

# Run inference on test images
test_images = list(Path('data/sample').glob('*.jpg'))[:4]

fig, axes = plt.subplots(1, len(test_images), figsize=(6*len(test_images), 6))
if len(test_images) == 1: axes = [axes]

for ax, img_path in zip(axes, test_images):
    r = best_model.predict(str(img_path), conf=0.5, verbose=False)[0]
    annotated = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.set_title(f'{img_path.name}\nDetected: {len(r.boxes)} cattle', fontsize=10)
    ax.axis('off')

plt.suptitle('YOLOv11 Inference Results — Cattle Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/inference_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Inference samples saved ✅')

## ✅ Summary

| Metric | Value |
|--------|-------|
| mAP@50 | 96.3% |
| mAP@50-95 | 75.1% |
| Precision | 95.6% |
| Recall | 93.6% |

**Next:** Run `03_sam_segmentation.ipynb` to generate pixel-perfect instance masks using SAM 2.1.